<div style="
    text-align: center; 
    background: linear-gradient(135deg, #0062ff 0%, #00d4ff 100%); 
    font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; 
    color: white; 
    padding: 35px 20px; 
    border-radius: 15px; 
    box-shadow: 0 10px 25px rgba(0, 98, 255, 0.3);
    margin-bottom: 25px;">
    <div style="font-size: 35px; font-weight: 800; letter-spacing: 1.5px; text-transform: uppercase; line-height: 1.2;">
        Trực Quan Hóa Dữ Liệu - Lab 02
    </div>
    <div style="font-size: 16px; font-weight: 500; margin-top: 10px; font-style: italic; opacity: 0.9;">
        "Khai thác và trực quan hóa dữ liệu bằng Tableau"
    </div>
    <div style="font-size: 18px; font-weight: 600; margin-top: 15px; border-top: 1px solid rgba(255,255,255,0.4); display: inline-block; padding-top: 10px; letter-spacing: 1px;">
        NHÓM 05 - FIT-HCMUS
    </div>
</div>

<div style="text-align: center; font-size: 40px; font-weight: bold;">
  TIỀN XỬ LÝ BỘ DỮ LIỆU TỪ WORLD BANK
</div>

## **1. NHẬP DỮ LIỆU VÀ KIỂM TRA TỔNG QUAN**

### **1.1. Mục tiêu**

Bước đầu tiên trong quy trình tiền xử lý là đọc dữ liệu thô từ World Development Indicators, kiểm tra cấu trúc file, phát hiện các dòng thừa, kiểm tra encoding, kiểm tra phạm vi năm, số lượng quốc gia/vùng lãnh thổ và số lượng chỉ số phát triển.

Bộ dữ liệu được tải từ World Bank dưới dạng bảng rộng, trong đó mỗi dòng tương ứng với một cặp **quốc gia/vùng lãnh thổ - chỉ số**, còn các cột năm từ `2000 [YR2000]` đến `2022 [YR2022]` chứa giá trị quan sát theo thời gian.

### **1.2. Ghi chú ban đầu về file dữ liệu**

Qua kiểm tra sơ bộ:

* `Data.csv` là file dữ liệu chính, chứa các giá trị chỉ số theo quốc gia/vùng lãnh thổ và theo năm.
* `Metadata.xlsx` là file metadata tải riêng từ World Bank, gồm nhiều sheet:
    * `Series - Metadata`: metadata mô tả các chỉ số.
    * `Country - Metadata`: metadata mô tả quốc gia/vùng lãnh thổ.
    * `Country-Series - Metadata`: ghi chú theo từng cặp quốc gia - chỉ số.
    * `FootNote`: ghi chú theo từng quốc gia - chỉ số - năm.
* Trong phạm vi tiền xử lý chính, nhóm sử dụng `Series - Metadata` để giải thích ý nghĩa chỉ số và `Country - Metadata` để bổ sung thông tin khu vực, nhóm thu nhập.
* Dữ liệu hiện có phạm vi năm **2000–2022**.

In [1]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 1: IMPORT THƯ VIỆN VÀ THIẾT LẬP ĐƯỜNG DẪN
# ─────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Xác định thư mục gốc project LAB02
# Notebook đang nằm trong LAB02/notebooks/
PROJECT_ROOT = Path.cwd()

# Nếu đang chạy notebook từ thư mục notebooks thì đi lên 1 cấp
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

DATA_PATH = RAW_DATA_DIR / "Data.csv"
METADATA_PATH = RAW_DATA_DIR / "Metadata.xlsx"

OUTPUT_DIR = PROCESSED_DATA_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Thư mục project        :", PROJECT_ROOT)
print("Đường dẫn Data.csv     :", DATA_PATH)
print("Đường dẫn Metadata.xlsx:", METADATA_PATH)
print("Thư mục xuất dữ liệu   :", OUTPUT_DIR)

# Kiểm tra file có tồn tại không
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy Data.csv tại: {DATA_PATH}")

if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy Metadata.xlsx tại: {METADATA_PATH}")

Thư mục project        : d:\BaiTapKi2Nam3\TQHDL\Lab\Lab02
Đường dẫn Data.csv     : d:\BaiTapKi2Nam3\TQHDL\Lab\Lab02\data\raw\Data.csv
Đường dẫn Metadata.xlsx: d:\BaiTapKi2Nam3\TQHDL\Lab\Lab02\data\raw\Metadata.xlsx
Thư mục xuất dữ liệu   : d:\BaiTapKi2Nam3\TQHDL\Lab\Lab02\data\processed


In [2]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 1.1: ĐỌC DỮ LIỆU CHÍNH VÀ METADATA
# ─────────────────────────────────────────────────────────────

def read_csv_with_fallback(path, encodings=("utf-8-sig", "utf-8", "cp1252", "latin1")):
    """
    Đọc file CSV với nhiều encoding khác nhau.
    File Data.csv của World Bank thường đọc tốt bằng utf-8-sig,
    nhưng vẫn dùng fallback để tránh lỗi encoding.
    """
    last_error = None
    
    for enc in encodings:
        try:
            df = pd.read_csv(path, dtype=str, encoding=enc)
            print(f"Đọc thành công {path.name} với encoding: {enc}")
            return df, enc
        except UnicodeDecodeError as e:
            last_error = e
    
    raise UnicodeDecodeError(
        "Không đọc được file với các encoding đã thử.",
        b"",
        0,
        1,
        str(last_error)
    )

# Đọc dữ liệu chính
df_data_raw, data_encoding = read_csv_with_fallback(DATA_PATH)

# Đọc metadata từ file Excel tải riêng từ World Bank
metadata_series_raw = pd.read_excel(
    METADATA_PATH,
    sheet_name="Series - Metadata",
    dtype=str
)

metadata_country_raw = pd.read_excel(
    METADATA_PATH,
    sheet_name="Country - Metadata",
    dtype=str
)

metadata_country_series_raw = pd.read_excel(
    METADATA_PATH,
    sheet_name="Country-Series - Metadata",
    dtype=str
)

metadata_footnote_raw = pd.read_excel(
    METADATA_PATH,
    sheet_name="FootNote",
    dtype=str
)

print("\n--- Kích thước dữ liệu thô")
print(f"Data.csv                    : {df_data_raw.shape}")
print(f"Series - Metadata           : {metadata_series_raw.shape}")
print(f"Country - Metadata          : {metadata_country_raw.shape}")
print(f"Country-Series - Metadata   : {metadata_country_series_raw.shape}")
print(f"FootNote                    : {metadata_footnote_raw.shape}")

print("\n--- 5 dòng đầu của Data.csv")
display(df_data_raw.head())

print("\n--- 8 dòng cuối của Data.csv")
display(df_data_raw.tail(8))

print("\n--- 5 dòng đầu của Series - Metadata")
display(metadata_series_raw.head())

print("\n--- 5 dòng đầu của Country - Metadata")
display(metadata_country_raw.head())

Đọc thành công Data.csv với encoding: utf-8-sig

--- Kích thước dữ liệu thô
Data.csv                    : (3197, 27)
Series - Metadata           : (12, 16)
Country - Metadata          : (265, 29)
Country-Series - Metadata   : (417, 4)
FootNote                    : (7009, 5)

--- 5 dòng đầu của Data.csv


,Country Name,Country Code,Series Name,Series Code,2000 [YR2000],2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],2006 [YR2006],2007 [YR2007],2008 [YR2008],2009 [YR2009],2010 [YR2010],2011 [YR2011],2012 [YR2012],2013 [YR2013],2014 [YR2014],2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022]
0,Afghanistan,AFG,GDP per capita (constant 2015 US$),NY.GDP.PCAP.KD,308.318269746638,277.118051443941,338.139973643387,346.071627096223,338.637273888197,363.640141436773,367.75831169358,410.757728879097,417.647282647498,488.830652491949,542.871030476037,525.42698276582,568.929021458341,580.603833333096,575.146245808546,565.569730408751,563.872336723147,562.769574140988,553.125151688293,557.861533207459,527.834554499306,408.625855217403,377.665627080705
1,Afghanistan,AFG,GDP per capita growth (annual %),NY.GDP.PCAP.KD.ZG,..,-10.1194841059326,22.0201902696299,2.34567163632688,-2.14821228495553,7.38337728197978,1.13248505528998,11.6923032922081,1.6772791560615,17.0438963216074,11.0550305527288,-3.21329500579925,8.27936899310519,2.0520682606117,-0.939984755736006,-1.66505744749013,-0.300121027406746,-0.195569548342789,-1.71374269254262,0.856294729087764,-5.38251464221077,-22.5844818732987,-7.57666891152213
2,Afghanistan,AFG,Poverty headcount ratio at national poverty li...,SI.POV.NAHC,..,..,..,..,..,..,..,33.7,..,..,..,38.3,..,..,..,..,54.5,..,..,47.1,..,..,..
3,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,55.005,55.511,56.225,57.171,57.81,58.247,58.553,58.956,59.708,60.248,60.702,61.25,61.735,62.188,62.26,62.27,62.646,62.406,62.443,62.941,61.454,60.417,65.617
4,Afghanistan,AFG,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,..,..,16.70697403,17.74602509,21.42300415,25.11388779,28.9412632,32.70893097,39.88624954,43.13351822,46.42438889,52.18721771,52.45247269,56.16043472,60.04951096,59.91960526,61.37234497,66.82388306,71.22509003,74.0642395,80.0892334,81.52112579,80.6516037



--- 8 dòng cuối của Data.csv


,Country Name,Country Code,Series Name,Series Code,2000 [YR2000],2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],2006 [YR2006],2007 [YR2007],2008 [YR2008],2009 [YR2009],2010 [YR2010],2011 [YR2011],2012 [YR2012],2013 [YR2013],2014 [YR2014],2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022]
3189,World,WLD,"School enrollment, secondary (% gross)",SE.SEC.ENRR,59.471851348877,60.2711296081543,61.6164588928223,62.3989791870117,63.4511299133301,64.4113006591797,65.3684387207031,66.8459777832031,68.3432464599609,69.3999710083008,70.9471435546875,71.9032669067383,72.7067337036133,74.0367202758789,74.9258728027344,74.9668884277344,75.1899871826172,74.9489288330078,75.54248046875,75.9502716064453,76.5068206787109,77.296272277832,77.5006103515625
3190,World,WLD,"School enrollment, tertiary (% gross)",SE.TER.ENRR,19.484260559082,20.3930397033691,21.6429405212402,22.7504901885986,23.6235904693604,24.3295593261719,25.7015895843506,26.303430557251,27.350980758667,28.2910709381104,29.5784606933594,31.1437301635742,32.3113899230957,33.0915985107422,35.5645713806152,36.6130981445312,37.1431999206543,37.4994316101074,37.8525314331055,38.6688690185547,39.7074089050293,41.0742492675781,42.0465698242188
3191,World,WLD,"School enrollment, primary and secondary (gros...",SE.ENR.PRSC.FM.ZS,0.92127001285553,0.924700021743774,0.927450001239777,0.944840013980865,0.945909976959229,0.950500011444092,0.954280018806458,0.958429992198944,0.965240001678467,0.972249984741211,0.97146999835968,0.973749995231628,0.977349996566772,0.991919994354248,0.994080007076263,0.995039999485016,0.999909996986389,0.998250007629395,0.987030029296875,0.988049983978271,0.985639989376068,..,..
3192,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3193,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3194,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3195,Data from database: World Development Indicators,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3196,Last Updated: 04/08/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- 5 dòng đầu của Series - Metadata


,Code,License Type,Indicator Name,Long definition,Source,Topic,Dataset,Unit of measure,Periodicity,Reference period,Aggregation method,Statistical concept and methodology,Development relevance,Limitations and exceptions,Other notes,License URL
0,NY.GDP.PCAP.KD,CC BY-4.0,GDP per capita (constant 2015 US$),Gross domestic product is the total income ear...,"Country official statistics, National Statisti...",Economic Policy & Debt: National accounts: US$...,WB_WDI,constant 2015 US$,Annual,1960-2024,Weighted average,Methodology: National accounts are compiled in...,This indicator is related to the national acco...,NaN,NaN,https://datacatalog.worldbank.org/public-licen...
1,NY.GDP.PCAP.KD.ZG,CC BY-4.0,GDP per capita (annual % growth),Gross domestic product is the total income ear...,"Country official statistics, National Statisti...",Economic Policy & Debt: National accounts: Gro...,WB_WDI,%,Annual,1961-2024,Weighted average,Methodology: National accounts are compiled in...,This indicator is related to the national acco...,NaN,NaN,https://datacatalog.worldbank.org/public-licen...
2,SI.POV.NAHC,CC BY-4.0,Poverty headcount ratio at national poverty li...,National poverty headcount ratio is the percen...,"World Bank, Poverty and Inequality Platform. D...",Poverty: Poverty rates,WB_WDI,%,Annual,1985-2025,Population-weighted average,Methodology: Poverty headcount ratio among the...,The poverty rate as defined by national povert...,NaN,This series only includes estimates that to th...,https://datacatalog.worldbank.org/public-licen...
3,SP.DYN.LE00.IN,CC BY-4.0,"Life expectancy at birth, total (years)",Life expectancy at birth indicates the number ...,"World Population Prospects, United Nations (UN...",Health: Mortality,WB_WDI,Years,Annual,1960-2024,Weighted average,Methodology: Life expectancy at birth is deriv...,Mortality rates for different age groups (infa...,Annual data series from United Nations Populat...,NaN,https://datacatalog.worldbank.org/public-licen...
4,SH.XPD.CHEX.PC.CD,CC BY-4.0,Current health expenditure per capita (current...,Current expenditures on health per capita in c...,"Global Health Expenditure Database, updated De...",Health: Health systems,WB_WDI,Current US$,Annual,2000-2024,Weighted average,Methodology: The Health SHA 2011 tracks all he...,Strengthening health financing is one objectiv...,NaN,NaN,https://datacatalog.worldbank.org/public-licen...



--- 5 dòng đầu của Country - Metadata


,Code,Long Name,Income Group,Region,Lending category,Other groups,Currency Unit,Latest population census,Latest household survey,Special Notes,National accounts base year,National accounts reference year,System of National Accounts,SNA price valuation,PPP survey years,Balance of Payments Manual in use,External debt Reporting status,System of trade,Government Accounting concept,IMF data dissemination standard,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,2-alpha code,WB-2 code,Table Name,Short Name
0,AFG,Islamic State of Afghanistan,Low income,Middle East & North Africa,IDA,HIPC,Afghan afghani,1979,Multiple Indicator Cluster Survey 2022-2023,The reporting period for national accounts dat...,2016,NaN,Country uses the 2008 System of National Accou...,Value added at basic prices (VAB),NaN,BPM6,Estimate,General trade system,Consolidated central government,Enhanced General Data Dissemination System (e-...,"Integrated household survey (IHS), 2016/17",NaN,NaN,NaN,2018,AF,AF,Afghanistan,Afghanistan
1,ALB,Republic of Albania,Upper middle income,Europe & Central Asia,IBRD,NaN,Albanian lek,2023,Demographic and Health Survey 2017 - 2018,NaN,Original chained constant price data are resca...,2020,Country uses the 2008 System of National Accou...,Value added at basic prices (VAB),Rolling surveys (annual estimation),BPM6,Actual,Special trade system,Consolidated central government,Enhanced General Data Dissemination System (e-...,Living Standards Measurement Study Survey (LSM...,Yes,2012,2013,2018,AL,AL,Albania,Albania
2,DZA,People's Democratic Republic of Algeria,Upper middle income,Middle East & North Africa,IBRD,NaN,Algerian dinar,2022,Multiple Indicator Cluster Survey 2018-2019,NaN,Original chained constant price data are resca...,2001,Country uses the 2008 System of National Accou...,Value added at basic prices (VAB),"2021, 2017, 2011",BPM6,Actual,Special trade system,Consolidated central government,Enhanced General Data Dissemination System (e-...,"Integrated household survey (IHS), 2011",NaN,NaN,2010,2017,DZ,DZ,Algeria,Algeria
3,ASM,American Samoa,High income,East Asia & Pacific,NaN,NaN,U.S. dollar,2020 (expected),NaN,NaN,Original chained constant price data are resca...,2012,Country uses the 2008 System of National Accou...,Value added at basic prices (VAB),2011 (Only for individual consumption expendit...,NaN,NaN,NaN,NaN,NaN,NaN,Yes,2008,NaN,NaN,AS,AS,American Samoa,American Samoa
4,AND,Principality of Andorra,High income,Europe & Central Asia,NaN,NaN,Euro,2021,NaN,NaN,2010,NaN,Country uses the 1993 System of National Accou...,Value added at basic prices (VAB),NaN,NaN,NaN,General trade system,NaN,NaN,NaN,Yes,NaN,NaN,2018,AD,AD,Andorra,Andorra


### **1.3. Kiểm tra cấu trúc dữ liệu thô**

File `Data.csv` đang ở dạng bảng rộng:

* 4 cột định danh chính:
    * `Country Name`
    * `Country Code`
    * `Series Name`
    * `Series Code`
* Các cột năm:
    * `2000 [YR2000]`
    * ...
    * `2022 [YR2022]`

Trong dữ liệu gốc, World Bank dùng ký hiệu `..` để biểu diễn giá trị thiếu. Các giá trị này cần được chuyển thành `NaN` trước khi ép kiểu số.

Ngoài ra, cuối file có các dòng footer không phải quan sát dữ liệu thật, nên cần loại bỏ.

In [3]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 1.2: KIỂM TRA CẤU TRÚC VÀ PHÁT HIỆN CỘT NĂM
# ─────────────────────────────────────────────────────────────

ID_COLS_RAW = ["Country Name", "Country Code", "Series Name", "Series Code"]

# Tự động nhận diện các cột năm dạng "2000 [YR2000]"
YEAR_COLS_RAW = [
    col for col in df_data_raw.columns
    if re.match(r"^\d{4}\s+\[YR\d{4}\]$", str(col))
]

print("--- Danh sách cột định danh")
print(ID_COLS_RAW)

print("\n--- Số lượng cột năm")
print(len(YEAR_COLS_RAW))

print("\n--- Phạm vi năm trong file")
years_detected = [int(re.search(r"\d{4}", col).group()) for col in YEAR_COLS_RAW]
print(min(years_detected), "→", max(years_detected))

print("\n--- Danh sách cột năm")
print(YEAR_COLS_RAW)

print("\n--- Kiểm tra các cột bắt buộc")
missing_required_cols = [col for col in ID_COLS_RAW if col not in df_data_raw.columns]
if missing_required_cols:
    print("Thiếu cột bắt buộc:", missing_required_cols)
else:
    print("Đủ các cột định danh bắt buộc.")

--- Danh sách cột định danh
['Country Name', 'Country Code', 'Series Name', 'Series Code']

--- Số lượng cột năm
23

--- Phạm vi năm trong file
2000 → 2022

--- Danh sách cột năm
['2000 [YR2000]', '2001 [YR2001]', '2002 [YR2002]', '2003 [YR2003]', '2004 [YR2004]', '2005 [YR2005]', '2006 [YR2006]', '2007 [YR2007]', '2008 [YR2008]', '2009 [YR2009]', '2010 [YR2010]', '2011 [YR2011]', '2012 [YR2012]', '2013 [YR2013]', '2014 [YR2014]', '2015 [YR2015]', '2016 [YR2016]', '2017 [YR2017]', '2018 [YR2018]', '2019 [YR2019]', '2020 [YR2020]', '2021 [YR2021]', '2022 [YR2022]']

--- Kiểm tra các cột bắt buộc
Đủ các cột định danh bắt buộc.


## **2. LÀM SẠCH CẤU TRÚC DỮ LIỆU**

### **2.1. Loại bỏ dòng thừa và chuẩn hóa kiểu dữ liệu**

Dữ liệu thô có một số dòng cuối file không phải quan sát dữ liệu, bao gồm dòng trống và dòng ghi chú từ World Bank. Các dòng này không có đầy đủ `Country Code`, `Series Name`, `Series Code`, do đó có thể loại bỏ an toàn.

Sau đó, các cột năm được xử lý theo quy tắc:

* Chuyển ký hiệu `..` thành `NaN`.
* Ép kiểu các cột năm về dạng số thực.
* Giữ nguyên giá trị thiếu thay vì tự ý điền trung bình hoặc nội suy.

Đối với dữ liệu phát triển quốc gia theo năm, việc tự ý điền thiếu có thể làm sai lệch xu hướng thật. Vì vậy, nhóm chỉ chuẩn hóa kiểu dữ liệu và tạo thêm bảng thống kê missing để phục vụ báo cáo.

In [4]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 2: LÀM SẠCH DATA.CSV
# ─────────────────────────────────────────────────────────────

df_wide = df_data_raw.copy()

# 1. Chuẩn hóa khoảng trắng trong tên cột
df_wide.columns = df_wide.columns.str.strip()

# 2. Chuẩn hóa chuỗi rỗng thành NaN
df_wide = df_wide.replace(r"^\s*$", np.nan, regex=True)

# 3. Loại bỏ các dòng không có đủ 4 cột định danh
before_rows = len(df_wide)
df_wide = df_wide.dropna(subset=ID_COLS_RAW, how="any").copy()
after_drop_id_rows = len(df_wide)

# 4. Loại bỏ các dòng footer nếu còn sót
footer_mask = df_wide["Country Name"].astype(str).str.startswith("Data from database", na=False) | \
              df_wide["Country Name"].astype(str).str.startswith("Last Updated", na=False)

df_wide = df_wide[~footer_mask].copy()
after_drop_footer_rows = len(df_wide)

# 5. Chuẩn hóa khoảng trắng trong các cột định danh
for col in ID_COLS_RAW:
    df_wide[col] = df_wide[col].astype(str).str.strip()

# 6. Chuyển ".." thành NaN và ép kiểu số cho các cột năm
for col in YEAR_COLS_RAW:
    df_wide[col] = (
        df_wide[col]
        .replace("..", np.nan)
        .pipe(pd.to_numeric, errors="coerce")
    )

print("--- Kết quả làm sạch cấu trúc")
print(f"Số dòng ban đầu                         : {before_rows}")
print(f"Số dòng sau khi drop dòng thiếu định danh: {after_drop_id_rows}")
print(f"Số dòng sau khi drop footer              : {after_drop_footer_rows}")

print("\n--- Kích thước dữ liệu sau làm sạch")
print(df_wide.shape)

print("\n--- Kiểu dữ liệu sau xử lý")
display(df_wide.dtypes.to_frame("dtype").T)

print("\n--- 5 dòng đầu sau làm sạch")
display(df_wide.head())

--- Kết quả làm sạch cấu trúc
Số dòng ban đầu                         : 3197
Số dòng sau khi drop dòng thiếu định danh: 3192
Số dòng sau khi drop footer              : 3192

--- Kích thước dữ liệu sau làm sạch
(3192, 27)

--- Kiểu dữ liệu sau xử lý


,Country Name,Country Code,Series Name,Series Code,2000 [YR2000],2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],2006 [YR2006],2007 [YR2007],2008 [YR2008],2009 [YR2009],2010 [YR2010],2011 [YR2011],2012 [YR2012],2013 [YR2013],2014 [YR2014],2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022]
dtype,object,object,object,object,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64



--- 5 dòng đầu sau làm sạch


,Country Name,Country Code,Series Name,Series Code,2000 [YR2000],2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],2006 [YR2006],2007 [YR2007],2008 [YR2008],2009 [YR2009],2010 [YR2010],2011 [YR2011],2012 [YR2012],2013 [YR2013],2014 [YR2014],2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022]
0,Afghanistan,AFG,GDP per capita (constant 2015 US$),NY.GDP.PCAP.KD,308.318,277.118,338.140,346.072,338.637,363.640,367.758,410.758,417.647,488.831,542.871,525.427,568.929,580.604,575.146,565.570,563.872,562.770,553.125,557.862,527.835,408.626,377.666
1,Afghanistan,AFG,GDP per capita growth (annual %),NY.GDP.PCAP.KD.ZG,NaN,-10.119,22.020,2.346,-2.148,7.383,1.132,11.692,1.677,17.044,11.055,-3.213,8.279,2.052,-0.940,-1.665,-0.300,-0.196,-1.714,0.856,-5.383,-22.584,-7.577
2,Afghanistan,AFG,Poverty headcount ratio at national poverty li...,SI.POV.NAHC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.700,NaN,NaN,NaN,38.300,NaN,NaN,NaN,NaN,54.500,NaN,NaN,47.100,NaN,NaN,NaN
3,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,55.005,55.511,56.225,57.171,57.810,58.247,58.553,58.956,59.708,60.248,60.702,61.250,61.735,62.188,62.260,62.270,62.646,62.406,62.443,62.941,61.454,60.417,65.617
4,Afghanistan,AFG,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,NaN,NaN,16.707,17.746,21.423,25.114,28.941,32.709,39.886,43.134,46.424,52.187,52.452,56.160,60.050,59.920,61.372,66.824,71.225,74.064,80.089,81.521,80.652


In [5]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 2.1: KIỂM TRA TÍNH TOÀN VẸN CỦA DỮ LIỆU
# ─────────────────────────────────────────────────────────────

n_countries = df_wide["Country Code"].nunique()
n_indicators = df_wide["Series Code"].nunique()
n_years = len(YEAR_COLS_RAW)

expected_rows = n_countries * n_indicators
actual_rows = len(df_wide)

duplicate_keys = df_wide.duplicated(subset=["Country Code", "Series Code"]).sum()
rows_all_year_missing = df_wide[YEAR_COLS_RAW].isna().all(axis=1).sum()

print("=" * 70)
print("KIỂM TRA TỔNG QUAN DỮ LIỆU SAU LÀM SẠCH")
print("=" * 70)

print(f"Số quốc gia/vùng/tổng hợp       : {n_countries}")
print(f"Số chỉ số phát triển             : {n_indicators}")
print(f"Số năm quan sát                  : {n_years}")
print(f"Số dòng kỳ vọng                  : {expected_rows}")
print(f"Số dòng thực tế                  : {actual_rows}")
print(f"Số khóa Country Code + Series Code bị trùng: {duplicate_keys}")
print(f"Số dòng thiếu toàn bộ các năm    : {rows_all_year_missing}")

if expected_rows == actual_rows and duplicate_keys == 0:
    print("\nDữ liệu có cấu trúc nhất quán: mỗi quốc gia/vùng có đủ dòng cho mỗi chỉ số.")
else:
    print("\nCần kiểm tra thêm vì số dòng thực tế không khớp cấu trúc kỳ vọng.")

print("\n--- Danh sách 12 chỉ số trong dữ liệu")
indicator_list = (
    df_wide[["Series Code", "Series Name"]]
    .drop_duplicates()
    .sort_values("Series Code")
    .reset_index(drop=True)
)
display(indicator_list)

KIỂM TRA TỔNG QUAN DỮ LIỆU SAU LÀM SẠCH
Số quốc gia/vùng/tổng hợp       : 266
Số chỉ số phát triển             : 12
Số năm quan sát                  : 23
Số dòng kỳ vọng                  : 3192
Số dòng thực tế                  : 3192
Số khóa Country Code + Series Code bị trùng: 0
Số dòng thiếu toàn bộ các năm    : 273

Dữ liệu có cấu trúc nhất quán: mỗi quốc gia/vùng có đủ dòng cho mỗi chỉ số.

--- Danh sách 12 chỉ số trong dữ liệu


,Series Code,Series Name
0,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...
1,EN.GHG.CO2.PC.CE.AR5,Carbon dioxide (CO2) emissions excluding LULUC...
2,NY.GDP.PCAP.KD,GDP per capita (constant 2015 US$)
3,NY.GDP.PCAP.KD.ZG,GDP per capita growth (annual %)
4,SE.ENR.PRSC.FM.ZS,"School enrollment, primary and secondary (gros..."
5,SE.PRM.ENRR,"School enrollment, primary (% gross)"
6,SE.SEC.ENRR,"School enrollment, secondary (% gross)"
7,SE.TER.ENRR,"School enrollment, tertiary (% gross)"
8,SH.DYN.MORT,"Mortality rate, under-5 (per 1,000 live births)"
9,SH.XPD.CHEX.PC.CD,Current health expenditure per capita (current...


## **3. TÁCH VÀ CHUẨN HÓA METADATA**

### **3.1. Metadata chỉ số và metadata quốc gia**
Nhóm sử dụng hai sheet chính:

* `Series - Metadata`: mô tả các chỉ số phát triển, bao gồm mã chỉ số, tên chỉ số, định nghĩa, chủ đề, đơn vị đo, chu kỳ cập nhật, phương pháp tổng hợp và nguồn dữ liệu.
* `Country - Metadata`: mô tả quốc gia/vùng lãnh thổ, bao gồm tên đầy đủ, nhóm thu nhập, khu vực, đơn vị tiền tệ và một số thông tin thống kê bổ sung.

Việc dùng metadata giúp dữ liệu sau tiền xử lý có thêm ngữ cảnh, thuận tiện hơn cho việc xây dựng dashboard trong Tableau và diễn giải kết quả trong báo cáo.

In [6]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 3: CHUẨN HÓA METADATA CHỈ SỐ VÀ METADATA QUỐC GIA
# ─────────────────────────────────────────────────────────────

# 3.1. Chuẩn hóa metadata chỉ số
indicator_metadata = metadata_series_raw.copy()
indicator_metadata.columns = indicator_metadata.columns.str.strip()
indicator_metadata = indicator_metadata.replace(r"^\s*$", np.nan, regex=True)

rename_indicator_cols = {
    "Code": "indicator_code",
    "Indicator Name": "indicator_name_en",
    "Long definition": "long_definition",
    "Source": "source",
    "Topic": "topic",
    "Dataset": "dataset",
    "Unit of measure": "unit_of_measure",
    "Periodicity": "periodicity",
    "Reference period": "reference_period",
    "Aggregation method": "aggregation_method",
    "Statistical concept and methodology": "methodology",
    "Development relevance": "development_relevance",
    "Limitations and exceptions": "limitations",
    "Other notes": "other_notes",
    "License URL": "license_url",
    "License Type": "license_type"
}

indicator_metadata = indicator_metadata.rename(columns=rename_indicator_cols)

valid_indicator_codes = set(df_wide["Series Code"].unique())

indicator_metadata = indicator_metadata[
    indicator_metadata["indicator_code"].isin(valid_indicator_codes)
].copy()

indicator_metadata = indicator_metadata.drop_duplicates(subset=["indicator_code"])

print("--- Metadata chỉ số sau khi chuẩn hóa")
print(indicator_metadata.shape)

display(indicator_metadata.head())


# 3.2. Chuẩn hóa metadata quốc gia/vùng lãnh thổ
country_metadata = metadata_country_raw.copy()
country_metadata.columns = country_metadata.columns.str.strip()
country_metadata = country_metadata.replace(r"^\s*$", np.nan, regex=True)

rename_country_cols = {
    "Code": "country_code",
    "Long Name": "country_long_name",
    "Income Group": "income_group",
    "Region": "region",
    "Lending category": "lending_category",
    "Other groups": "other_groups",
    "Currency Unit": "currency_unit",
    "Latest population census": "latest_population_census",
    "Latest household survey": "latest_household_survey",
    "Special Notes": "country_special_notes",
    "Table Name": "country_table_name",
    "Short Name": "country_short_name",
    "2-alpha code": "country_alpha2_code",
    "WB-2 code": "country_wb2_code"
}

country_metadata = country_metadata.rename(columns=rename_country_cols)

country_keep_cols = [
    "country_code",
    "country_long_name",
    "country_short_name",
    "country_table_name",
    "region",
    "income_group",
    "lending_category",
    "other_groups",
    "currency_unit",
    "country_alpha2_code",
    "country_wb2_code",
    "latest_population_census",
    "latest_household_survey",
    "country_special_notes"
]

country_keep_cols = [col for col in country_keep_cols if col in country_metadata.columns]

country_metadata = (
    country_metadata[country_keep_cols]
    .drop_duplicates(subset=["country_code"])
    .reset_index(drop=True)
)

print("\n--- Metadata quốc gia/vùng lãnh thổ sau khi chuẩn hóa")
print(country_metadata.shape)

display(country_metadata.head())

--- Metadata chỉ số sau khi chuẩn hóa
(12, 16)


,indicator_code,license_type,indicator_name_en,long_definition,source,topic,dataset,unit_of_measure,periodicity,reference_period,aggregation_method,methodology,development_relevance,limitations,other_notes,license_url
0,NY.GDP.PCAP.KD,CC BY-4.0,GDP per capita (constant 2015 US$),Gross domestic product is the total income ear...,"Country official statistics, National Statisti...",Economic Policy & Debt: National accounts: US$...,WB_WDI,constant 2015 US$,Annual,1960-2024,Weighted average,Methodology: National accounts are compiled in...,This indicator is related to the national acco...,NaN,NaN,https://datacatalog.worldbank.org/public-licen...
1,NY.GDP.PCAP.KD.ZG,CC BY-4.0,GDP per capita (annual % growth),Gross domestic product is the total income ear...,"Country official statistics, National Statisti...",Economic Policy & Debt: National accounts: Gro...,WB_WDI,%,Annual,1961-2024,Weighted average,Methodology: National accounts are compiled in...,This indicator is related to the national acco...,NaN,NaN,https://datacatalog.worldbank.org/public-licen...
2,SI.POV.NAHC,CC BY-4.0,Poverty headcount ratio at national poverty li...,National poverty headcount ratio is the percen...,"World Bank, Poverty and Inequality Platform. D...",Poverty: Poverty rates,WB_WDI,%,Annual,1985-2025,Population-weighted average,Methodology: Poverty headcount ratio among the...,The poverty rate as defined by national povert...,NaN,This series only includes estimates that to th...,https://datacatalog.worldbank.org/public-licen...
3,SP.DYN.LE00.IN,CC BY-4.0,"Life expectancy at birth, total (years)",Life expectancy at birth indicates the number ...,"World Population Prospects, United Nations (UN...",Health: Mortality,WB_WDI,Years,Annual,1960-2024,Weighted average,Methodology: Life expectancy at birth is deriv...,Mortality rates for different age groups (infa...,Annual data series from United Nations Populat...,NaN,https://datacatalog.worldbank.org/public-licen...
4,SH.XPD.CHEX.PC.CD,CC BY-4.0,Current health expenditure per capita (current...,Current expenditures on health per capita in c...,"Global Health Expenditure Database, updated De...",Health: Health systems,WB_WDI,Current US$,Annual,2000-2024,Weighted average,Methodology: The Health SHA 2011 tracks all he...,Strengthening health financing is one objectiv...,NaN,NaN,https://datacatalog.worldbank.org/public-licen...



--- Metadata quốc gia/vùng lãnh thổ sau khi chuẩn hóa
(265, 14)


,country_code,country_long_name,country_short_name,country_table_name,region,income_group,lending_category,other_groups,currency_unit,country_alpha2_code,country_wb2_code,latest_population_census,latest_household_survey,country_special_notes
0,AFG,Islamic State of Afghanistan,Afghanistan,Afghanistan,Middle East & North Africa,Low income,IDA,HIPC,Afghan afghani,AF,AF,1979,Multiple Indicator Cluster Survey 2022-2023,The reporting period for national accounts dat...
1,ALB,Republic of Albania,Albania,Albania,Europe & Central Asia,Upper middle income,IBRD,NaN,Albanian lek,AL,AL,2023,Demographic and Health Survey 2017 - 2018,NaN
2,DZA,People's Democratic Republic of Algeria,Algeria,Algeria,Middle East & North Africa,Upper middle income,IBRD,NaN,Algerian dinar,DZ,DZ,2022,Multiple Indicator Cluster Survey 2018-2019,NaN
3,ASM,American Samoa,American Samoa,American Samoa,East Asia & Pacific,High income,NaN,NaN,U.S. dollar,AS,AS,2020 (expected),NaN,NaN
4,AND,Principality of Andorra,Andorra,Andorra,Europe & Central Asia,High income,NaN,NaN,Euro,AD,AD,2021,NaN,NaN


### **3.2. Việt hóa tên chỉ số và phân nhóm phân tích**

Để thuận tiện khi trực quan hóa bằng Tableau và viết báo cáo, nhóm tạo thêm các trường mô tả bằng tiếng Việt cho từng chỉ số.

Các chỉ số được chia thành 4 nhóm chủ đề chính:

* **Kinh tế:** GDP bình quân đầu người, tăng trưởng GDP bình quân đầu người, tỷ lệ nghèo.
* **Sức khỏe:** Tuổi thọ, chi tiêu y tế, tử vong trẻ em dưới 5 tuổi.
* **Môi trường:** Phát thải CO2 bình quân đầu người, tỷ trọng năng lượng tái tạo.
* **Giáo dục:** Tỷ lệ nhập học các cấp và chỉ số cân bằng giới trong giáo dục.

Các trường này không làm thay đổi dữ liệu gốc mà chỉ bổ sung nhãn diễn giải phục vụ dashboard.

In [7]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 3.2: VIỆT HÓA TÊN CHỈ SỐ VÀ PHÂN NHÓM CHỦ ĐỀ
# ─────────────────────────────────────────────────────────────

INDICATOR_INFO_VI = {
    "NY.GDP.PCAP.KD": {
        "indicator_name_vi": "GDP bình quân đầu người (USD cố định 2015)",
        "indicator_group": "Kinh tế",
        "unit_vi": "USD cố định 2015",
        "short_name": "GDP/người",
        "tableau_field_name": "gdp_per_capita"
    },
    "NY.GDP.PCAP.KD.ZG": {
        "indicator_name_vi": "Tăng trưởng GDP bình quân đầu người (%)",
        "indicator_group": "Kinh tế",
        "unit_vi": "%",
        "short_name": "Tăng trưởng GDP/người",
        "tableau_field_name": "gdp_per_capita_growth_pct"
    },
    "SI.POV.NAHC": {
        "indicator_name_vi": "Tỷ lệ nghèo theo chuẩn quốc gia (%)",
        "indicator_group": "Kinh tế - Xã hội",
        "unit_vi": "%",
        "short_name": "Tỷ lệ nghèo",
        "tableau_field_name": "poverty_headcount_pct"
    },
    "SP.DYN.LE00.IN": {
        "indicator_name_vi": "Tuổi thọ trung bình khi sinh (năm)",
        "indicator_group": "Sức khỏe",
        "unit_vi": "Năm",
        "short_name": "Tuổi thọ",
        "tableau_field_name": "life_expectancy_years"
    },
    "SH.XPD.CHEX.PC.CD": {
        "indicator_name_vi": "Chi tiêu y tế bình quân đầu người (USD hiện hành)",
        "indicator_group": "Sức khỏe",
        "unit_vi": "USD hiện hành",
        "short_name": "Chi tiêu y tế/người",
        "tableau_field_name": "health_expenditure_pc_usd"
    },
    "SH.DYN.MORT": {
        "indicator_name_vi": "Tử vong trẻ em dưới 5 tuổi (trên 1.000 ca sinh sống)",
        "indicator_group": "Sức khỏe",
        "unit_vi": "Trên 1.000 ca sinh sống",
        "short_name": "Tử vong dưới 5 tuổi",
        "tableau_field_name": "under5_mortality_per_1000"
    },
    "EN.GHG.CO2.PC.CE.AR5": {
        "indicator_name_vi": "Phát thải CO2 bình quân đầu người (tCO2e/người)",
        "indicator_group": "Môi trường",
        "unit_vi": "tCO2e/người",
        "short_name": "CO2/người",
        "tableau_field_name": "co2_pc_tco2e"
    },
    "EG.FEC.RNEW.ZS": {
        "indicator_name_vi": "Tỷ trọng năng lượng tái tạo trong tiêu thụ năng lượng cuối cùng (%)",
        "indicator_group": "Môi trường",
        "unit_vi": "%",
        "short_name": "Năng lượng tái tạo",
        "tableau_field_name": "renewable_energy_pct"
    },
    "SE.PRM.ENRR": {
        "indicator_name_vi": "Tỷ lệ nhập học tiểu học, gross (%)",
        "indicator_group": "Giáo dục",
        "unit_vi": "%",
        "short_name": "Nhập học tiểu học",
        "tableau_field_name": "school_primary_gross_pct"
    },
    "SE.SEC.ENRR": {
        "indicator_name_vi": "Tỷ lệ nhập học trung học, gross (%)",
        "indicator_group": "Giáo dục",
        "unit_vi": "%",
        "short_name": "Nhập học trung học",
        "tableau_field_name": "school_secondary_gross_pct"
    },
    "SE.TER.ENRR": {
        "indicator_name_vi": "Tỷ lệ nhập học bậc đại học/cao đẳng, gross (%)",
        "indicator_group": "Giáo dục",
        "unit_vi": "%",
        "short_name": "Nhập học bậc cao",
        "tableau_field_name": "school_tertiary_gross_pct"
    },
    "SE.ENR.PRSC.FM.ZS": {
        "indicator_name_vi": "Chỉ số cân bằng giới trong nhập học tiểu học và trung học (GPI)",
        "indicator_group": "Giáo dục",
        "unit_vi": "Tỷ lệ nữ/nam",
        "short_name": "GPI giáo dục",
        "tableau_field_name": "education_gpi"
    }
}

indicator_info_vi = (
    pd.DataFrame.from_dict(INDICATOR_INFO_VI, orient="index")
    .reset_index()
    .rename(columns={"index": "indicator_code"})
)

indicator_metadata = indicator_metadata.merge(
    indicator_info_vi,
    on="indicator_code",
    how="left"
)

# Nếu metadata không có tên tiếng Anh vì đọc lỗi, lấy từ Data.csv để bổ sung
series_name_lookup = (
    df_wide[["Series Code", "Series Name"]]
    .drop_duplicates()
    .rename(columns={
        "Series Code": "indicator_code",
        "Series Name": "indicator_name_from_data"
    })
)

indicator_metadata = indicator_metadata.merge(
    series_name_lookup,
    on="indicator_code",
    how="left"
)

indicator_metadata["indicator_name_en"] = indicator_metadata["indicator_name_en"].fillna(
    indicator_metadata["indicator_name_from_data"]
)

indicator_metadata = indicator_metadata.drop(columns=["indicator_name_from_data"], errors="ignore")

print("--- Metadata sau khi bổ sung tên tiếng Việt và nhóm chỉ số")
display(indicator_metadata[[
    "indicator_code",
    "indicator_name_en",
    "indicator_name_vi",
    "indicator_group",
    "unit_vi",
    "short_name",
    "tableau_field_name"
]])

print("\n--- Kiểm tra số lượng chỉ số sau khi bổ sung metadata tiếng Việt")
print(indicator_metadata["indicator_code"].nunique(), "chỉ số")

print("\n--- Các chỉ số chưa có tên tiếng Việt nếu có")
missing_vi_name = indicator_metadata[indicator_metadata["indicator_name_vi"].isna()]
display(missing_vi_name[["indicator_code", "indicator_name_en"]])

--- Metadata sau khi bổ sung tên tiếng Việt và nhóm chỉ số


,indicator_code,indicator_name_en,indicator_name_vi,indicator_group,unit_vi,short_name,tableau_field_name
0,NY.GDP.PCAP.KD,GDP per capita (constant 2015 US$),GDP bình quân đầu người (USD cố định 2015),Kinh tế,USD cố định 2015,GDP/người,gdp_per_capita
1,NY.GDP.PCAP.KD.ZG,GDP per capita (annual % growth),Tăng trưởng GDP bình quân đầu người (%),Kinh tế,%,Tăng trưởng GDP/người,gdp_per_capita_growth_pct
2,SI.POV.NAHC,Poverty headcount ratio at national poverty li...,Tỷ lệ nghèo theo chuẩn quốc gia (%),Kinh tế - Xã hội,%,Tỷ lệ nghèo,poverty_headcount_pct
3,SP.DYN.LE00.IN,"Life expectancy at birth, total (years)",Tuổi thọ trung bình khi sinh (năm),Sức khỏe,Năm,Tuổi thọ,life_expectancy_years
4,SH.XPD.CHEX.PC.CD,Current health expenditure per capita (current...,Chi tiêu y tế bình quân đầu người (USD hiện hành),Sức khỏe,USD hiện hành,Chi tiêu y tế/người,health_expenditure_pc_usd
5,SH.DYN.MORT,"Mortality rate, under-5 (per 1,000 live births)",Tử vong trẻ em dưới 5 tuổi (trên 1.000 ca sinh...,Sức khỏe,Trên 1.000 ca sinh sống,Tử vong dưới 5 tuổi,under5_mortality_per_1000
6,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Môi trường,%,Năng lượng tái tạo,renewable_energy_pct
7,EN.GHG.CO2.PC.CE.AR5,Carbon dioxide (CO2) emissions excluding LULUC...,Phát thải CO2 bình quân đầu người (tCO2e/người),Môi trường,tCO2e/người,CO2/người,co2_pc_tco2e
8,SE.PRM.ENRR,"School enrollment, primary (% gross)","Tỷ lệ nhập học tiểu học, gross (%)",Giáo dục,%,Nhập học tiểu học,school_primary_gross_pct
9,SE.SEC.ENRR,"School enrollment, secondary (% gross)","Tỷ lệ nhập học trung học, gross (%)",Giáo dục,%,Nhập học trung học,school_secondary_gross_pct



--- Kiểm tra số lượng chỉ số sau khi bổ sung metadata tiếng Việt
12 chỉ số

--- Các chỉ số chưa có tên tiếng Việt nếu có


,indicator_code,indicator_name_en


## **4. CHUYỂN DỮ LIỆU TỪ DẠNG RỘNG SANG DẠNG DÀI**

### **4.1. Lý do chuyển đổi**

Dữ liệu tải từ World Bank đang ở dạng rộng, mỗi năm là một cột riêng. Dạng này thuận tiện để đọc thủ công nhưng chưa tối ưu cho Tableau.

Nhóm chuyển dữ liệu sang dạng dài với cấu trúc:

* `country_name`
* `country_code`
* `indicator_code`
* `indicator_name_en`
* `indicator_name_vi`
* `indicator_group`
* `year`
* `value`

Dạng dài phù hợp hơn cho:

* Vẽ biểu đồ chuỗi thời gian.
* Dùng filter theo năm, quốc gia, nhóm chỉ số.
* Xây dựng dashboard có selector/dropdown trong Tableau.
* So sánh linh hoạt nhiều chỉ số theo thời gian.

In [8]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 4: CHUYỂN DỮ LIỆU WIDE → LONG
# ─────────────────────────────────────────────────────────────

df_long = df_wide.melt(
    id_vars=ID_COLS_RAW,
    value_vars=YEAR_COLS_RAW,
    var_name="year_raw",
    value_name="value"
)

# Trích năm từ chuỗi "2000 [YR2000]"
df_long["year"] = df_long["year_raw"].str.extract(r"(\d{4})").astype(int)

# Đổi tên cột sang dạng snake_case
df_long = df_long.rename(columns={
    "Country Name": "country_name",
    "Country Code": "country_code",
    "Series Name": "indicator_name_en_raw",
    "Series Code": "indicator_code"
})

# Join metadata chỉ số
metadata_join_cols = [
    "indicator_code",
    "indicator_name_en",
    "indicator_name_vi",
    "indicator_group",
    "unit_vi",
    "short_name",
    "tableau_field_name",
    "long_definition",
    "topic",
    "source",
    "unit_of_measure",
    "periodicity",
    "reference_period",
    "aggregation_method"
]

metadata_join_cols = [col for col in metadata_join_cols if col in indicator_metadata.columns]

df_long = df_long.merge(
    indicator_metadata[metadata_join_cols],
    on="indicator_code",
    how="left"
)

# Nếu tên tiếng Anh từ metadata thiếu thì dùng tên trong Data.csv
df_long["indicator_name_en"] = df_long["indicator_name_en"].fillna(
    df_long["indicator_name_en_raw"]
)

# Join metadata quốc gia/vùng lãnh thổ
country_join_cols = [
    "country_code",
    "country_long_name",
    "country_short_name",
    "region",
    "income_group",
    "currency_unit"
]

country_join_cols = [col for col in country_join_cols if col in country_metadata.columns]

df_long = df_long.merge(
    country_metadata[country_join_cols],
    on="country_code",
    how="left"
)

# Tạo cột ngày để Tableau nhận diện time-series dễ hơn
df_long["year_date"] = pd.to_datetime(df_long["year"].astype(str) + "-01-01")

# Tạo giai đoạn để hỗ trợ filter/nhóm trong Tableau
def period_group(year):
    if 2000 <= year <= 2004:
        return "2000–2004"
    if 2005 <= year <= 2009:
        return "2005–2009"
    if 2010 <= year <= 2014:
        return "2010–2014"
    if 2015 <= year <= 2019:
        return "2015–2019"
    if 2020 <= year <= 2022:
        return "2020–2022"
    return "Khác"

df_long["period_group"] = df_long["year"].apply(period_group)

# Cờ dữ liệu thiếu
df_long["is_missing"] = df_long["value"].isna()
df_long["data_status"] = np.where(df_long["is_missing"], "Thiếu dữ liệu", "Có dữ liệu")

print("--- Kích thước dữ liệu dạng dài")
print(df_long.shape)

print("\n--- 5 dòng đầu")
display(df_long.head())

print("\n--- Tỉ lệ missing toàn bộ bảng long")
print(f"{df_long['value'].isna().mean() * 100:.2f}%")

print("\n--- Các mã quốc gia có trong Data.csv nhưng không có trong Country - Metadata")
missing_country_metadata = (
    df_long[df_long["country_long_name"].isna()][["country_name", "country_code"]]
    .drop_duplicates()
    .sort_values("country_code")
)

display(missing_country_metadata)

--- Kích thước dữ liệu dạng dài
(73416, 29)

--- 5 dòng đầu


,country_name,country_code,indicator_name_en_raw,indicator_code,year_raw,value,year,indicator_name_en,indicator_name_vi,indicator_group,unit_vi,short_name,tableau_field_name,long_definition,topic,source,unit_of_measure,periodicity,reference_period,aggregation_method,country_long_name,country_short_name,region,income_group,currency_unit,year_date,period_group,is_missing,data_status
0,Afghanistan,AFG,GDP per capita (constant 2015 US$),NY.GDP.PCAP.KD,2000 [YR2000],308.318,2000,GDP per capita (constant 2015 US$),GDP bình quân đầu người (USD cố định 2015),Kinh tế,USD cố định 2015,GDP/người,gdp_per_capita,Gross domestic product is the total income ear...,Economic Policy & Debt: National accounts: US$...,"Country official statistics, National Statisti...",constant 2015 US$,Annual,1960-2024,Weighted average,Islamic State of Afghanistan,Afghanistan,Middle East & North Africa,Low income,Afghan afghani,2000-01-01,2000–2004,False,Có dữ liệu
1,Afghanistan,AFG,GDP per capita growth (annual %),NY.GDP.PCAP.KD.ZG,2000 [YR2000],NaN,2000,GDP per capita (annual % growth),Tăng trưởng GDP bình quân đầu người (%),Kinh tế,%,Tăng trưởng GDP/người,gdp_per_capita_growth_pct,Gross domestic product is the total income ear...,Economic Policy & Debt: National accounts: Gro...,"Country official statistics, National Statisti...",%,Annual,1961-2024,Weighted average,Islamic State of Afghanistan,Afghanistan,Middle East & North Africa,Low income,Afghan afghani,2000-01-01,2000–2004,True,Thiếu dữ liệu
2,Afghanistan,AFG,Poverty headcount ratio at national poverty li...,SI.POV.NAHC,2000 [YR2000],NaN,2000,Poverty headcount ratio at national poverty li...,Tỷ lệ nghèo theo chuẩn quốc gia (%),Kinh tế - Xã hội,%,Tỷ lệ nghèo,poverty_headcount_pct,National poverty headcount ratio is the percen...,Poverty: Poverty rates,"World Bank, Poverty and Inequality Platform. D...",%,Annual,1985-2025,Population-weighted average,Islamic State of Afghanistan,Afghanistan,Middle East & North Africa,Low income,Afghan afghani,2000-01-01,2000–2004,True,Thiếu dữ liệu
3,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,2000 [YR2000],55.005,2000,"Life expectancy at birth, total (years)",Tuổi thọ trung bình khi sinh (năm),Sức khỏe,Năm,Tuổi thọ,life_expectancy_years,Life expectancy at birth indicates the number ...,Health: Mortality,"World Population Prospects, United Nations (UN...",Years,Annual,1960-2024,Weighted average,Islamic State of Afghanistan,Afghanistan,Middle East & North Africa,Low income,Afghan afghani,2000-01-01,2000–2004,False,Có dữ liệu
4,Afghanistan,AFG,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,2000 [YR2000],NaN,2000,Current health expenditure per capita (current...,Chi tiêu y tế bình quân đầu người (USD hiện hành),Sức khỏe,USD hiện hành,Chi tiêu y tế/người,health_expenditure_pc_usd,Current expenditures on health per capita in c...,Health: Health systems,"Global Health Expenditure Database, updated De...",Current US$,Annual,2000-2024,Weighted average,Islamic State of Afghanistan,Afghanistan,Middle East & North Africa,Low income,Afghan afghani,2000-01-01,2000–2004,True,Thiếu dữ liệu



--- Tỉ lệ missing toàn bộ bảng long
20.30%

--- Các mã quốc gia có trong Data.csv nhưng không có trong Country - Metadata


,country_name,country_code
3024,Not classified,INX


## **5. NHẬN DIỆN QUỐC GIA THẬT VÀ NHÓM TỔNG HỢP**

### **5.1. Vấn đề về đơn vị quan sát**

Dữ liệu WDI không chỉ bao gồm quốc gia/vùng lãnh thổ riêng lẻ mà còn có các nhóm tổng hợp như:

* `World`
* `High income`
* `Low income`
* `East Asia & Pacific`
* `European Union`
* `OECD members`
* `Sub-Saharan Africa`

Các nhóm tổng hợp này hữu ích khi so sánh bối cảnh chung, nhưng nếu phân tích xếp hạng quốc gia thì cần loại ra để tránh sai lệch.

Vì vậy, nhóm không xóa các dòng này khỏi dữ liệu chính, mà tạo thêm cột `is_aggregate` và `country_type` để có thể lọc linh hoạt trong Tableau.

In [9]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 5: ĐÁNH DẤU NHÓM TỔNG HỢP CỦA WORLD BANK
# ─────────────────────────────────────────────────────────────

# Danh sách mã tổng hợp phổ biến của World Bank trong WDI
# Không xóa khỏi dữ liệu chính, chỉ đánh dấu để lọc trong Tableau.
WB_AGGREGATE_CODES = {
    "AFE", "AFW", "ARB", "CSS", "CEB",
    "EAR", "EAS", "EAP", "TEA",
    "EMU", "ECS", "ECA", "TEC", "EUU",
    "FCS", "HPC",
    "HIC", "IBD", "IBT", "IDB", "IDX", "IDA",
    "LTE", "LCN", "LAC", "TLA", "LDC",
    "LMY", "LIC", "LMC", "MEA", "MNA", "TMN",
    "MIC", "NAC", "INX", "OED", "OSS", "PSS",
    "PST", "PRE", "SST", "SAS", "TSA",
    "SSF", "SSA", "TSS", "UMC", "WLD"
}

df_long["is_aggregate"] = df_long["country_code"].isin(WB_AGGREGATE_CODES)

df_long["country_type"] = np.where(
    df_long["is_aggregate"],
    "Nhóm tổng hợp/khu vực",
    "Quốc gia/vùng lãnh thổ"
)

print("--- Số lượng đơn vị theo loại")
display(
    df_long[["country_name", "country_code", "country_type"]]
    .drop_duplicates()
    .groupby("country_type")
    .size()
    .reset_index(name="so_luong")
)

print("\n--- Một số nhóm tổng hợp trong dữ liệu")
display(
    df_long[df_long["is_aggregate"]][["country_name", "country_code"]]
    .drop_duplicates()
    .sort_values("country_name")
    .head(30)
)

print("\n--- Một số quốc gia/vùng lãnh thổ thật")
display(
    df_long[~df_long["is_aggregate"]][["country_name", "country_code"]]
    .drop_duplicates()
    .sort_values("country_name")
    .head(30)
)

--- Số lượng đơn vị theo loại


,country_type,so_luong
0,Nhóm tổng hợp/khu vực,49
1,Quốc gia/vùng lãnh thổ,217



--- Một số nhóm tổng hợp trong dữ liệu


,country_name,country_code
2604,Africa Eastern and Southern,AFE
2616,Africa Western and Central,AFW
2628,Arab World,ARB
2640,Caribbean small states,CSS
2652,Central Europe and the Baltics,CEB
2664,Early-demographic dividend,EAR
2676,East Asia & Pacific,EAS
2700,East Asia & Pacific (IDA & IBRD countries),TEA
2688,East Asia & Pacific (excluding high income),EAP
2712,Euro area,EMU



--- Một số quốc gia/vùng lãnh thổ thật


,country_name,country_code
0,Afghanistan,AFG
12,Albania,ALB
24,Algeria,DZA
36,American Samoa,ASM
48,Andorra,AND
60,Angola,AGO
72,Antigua and Barbuda,ATG
84,Argentina,ARG
96,Armenia,ARM
108,Aruba,ABW


## **6. PHÂN TÍCH CHẤT LƯỢNG DỮ LIỆU**

### **6.1. Kiểm tra dữ liệu thiếu**

Dữ liệu World Development Indicators có mức độ thiếu khác nhau giữa các chỉ số. Đây là đặc điểm bình thường của dữ liệu phát triển quốc tế vì không phải quốc gia nào cũng công bố đầy đủ mọi chỉ số qua từng năm.

Nguyên tắc xử lý của nhóm:

* Không tự ý điền trung bình cho các chỉ số theo quốc gia.
* Không nội suy tuyến tính trên toàn bộ dữ liệu vì có thể tạo xu hướng giả.
* Giữ bảng đầy đủ có cả missing để kiểm tra chất lượng.
* Xuất bảng sạch không missing để đưa vào Tableau khi cần vẽ biểu đồ.
* Với các chỉ số thiếu nhiều như tỷ lệ nghèo, khi phân tích cần ghi rõ số lượng quan sát và tránh kết luận quá rộng.

In [10]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 6: THỐNG KÊ MISSING VALUE
# ─────────────────────────────────────────────────────────────

# 1. Missing theo chỉ số
missing_by_indicator = (
    df_long
    .groupby([
        "indicator_code",
        "indicator_name_vi",
        "indicator_group"
    ])
    .agg(
        total_observations=("value", "size"),
        missing_observations=("is_missing", "sum"),
        available_observations=("is_missing", lambda s: (~s).sum()),
        missing_pct=("is_missing", lambda s: s.mean() * 100)
    )
    .reset_index()
    .sort_values("missing_pct", ascending=False)
)

missing_by_indicator["missing_pct"] = missing_by_indicator["missing_pct"].round(2)

print("--- Tỉ lệ missing theo chỉ số")
display(missing_by_indicator)

# 2. Missing theo năm
missing_by_year = (
    df_long
    .groupby("year")
    .agg(
        total_observations=("value", "size"),
        missing_observations=("is_missing", "sum"),
        available_observations=("is_missing", lambda s: (~s).sum()),
        missing_pct=("is_missing", lambda s: s.mean() * 100)
    )
    .reset_index()
)

missing_by_year["missing_pct"] = missing_by_year["missing_pct"].round(2)

print("\n--- Tỉ lệ missing theo năm")
display(missing_by_year)

# 3. Missing theo chỉ số và năm
missing_by_indicator_year = (
    df_long
    .groupby([
        "indicator_code",
        "indicator_name_vi",
        "year"
    ])
    .agg(
        total_observations=("value", "size"),
        missing_observations=("is_missing", "sum"),
        available_observations=("is_missing", lambda s: (~s).sum()),
        missing_pct=("is_missing", lambda s: s.mean() * 100)
    )
    .reset_index()
)

missing_by_indicator_year["missing_pct"] = missing_by_indicator_year["missing_pct"].round(2)

print("\n--- Missing theo chỉ số và năm")
display(missing_by_indicator_year.head(20))

--- Tỉ lệ missing theo chỉ số


,indicator_code,indicator_name_vi,indicator_group,total_observations,missing_observations,available_observations,missing_pct
10,SI.POV.NAHC,Tỷ lệ nghèo theo chuẩn quốc gia (%),Kinh tế - Xã hội,6118,5083,1035,83.080
4,SE.ENR.PRSC.FM.ZS,Chỉ số cân bằng giới trong nhập học tiểu học v...,Giáo dục,6118,2281,3837,37.280
7,SE.TER.ENRR,"Tỷ lệ nhập học bậc đại học/cao đẳng, gross (%)",Giáo dục,6118,2118,4000,34.620
6,SE.SEC.ENRR,"Tỷ lệ nhập học trung học, gross (%)",Giáo dục,6118,1728,4390,28.240
5,SE.PRM.ENRR,"Tỷ lệ nhập học tiểu học, gross (%)",Giáo dục,6118,1237,4881,20.220
9,SH.XPD.CHEX.PC.CD,Chi tiêu y tế bình quân đầu người (USD hiện hành),Sức khỏe,6118,668,5450,10.920
8,SH.DYN.MORT,Tử vong trẻ em dưới 5 tuổi (trên 1.000 ca sinh...,Sức khỏe,6118,506,5612,8.270
0,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Môi trường,6118,408,5710,6.670
1,EN.GHG.CO2.PC.CE.AR5,Phát thải CO2 bình quân đầu người (tCO2e/người),Môi trường,6118,345,5773,5.640
2,NY.GDP.PCAP.KD,GDP bình quân đầu người (USD cố định 2015),Kinh tế,6118,257,5861,4.200



--- Tỉ lệ missing theo năm


,year,total_observations,missing_observations,available_observations,missing_pct
0,2000,3192,774,2418,24.250
1,2001,3192,694,2498,21.740
2,2002,3192,662,2530,20.740
3,2003,3192,671,2521,21.020
4,2004,3192,629,2563,19.710
5,2005,3192,614,2578,19.240
6,2006,3192,642,2550,20.110
7,2007,3192,626,2566,19.610
8,2008,3192,623,2569,19.520
9,2009,3192,613,2579,19.200



--- Missing theo chỉ số và năm


,indicator_code,indicator_name_vi,year,total_observations,missing_observations,available_observations,missing_pct
0,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,2000,266,11,255,4.140
1,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,2001,266,11,255,4.140
2,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,2002,266,9,257,3.380
3,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,2003,266,9,257,3.380
4,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,2004,266,9,257,3.380
5,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,2005,266,8,258,3.010
6,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,2006,266,8,258,3.010
7,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,2007,266,8,258,3.010
8,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,2008,266,8,258,3.010
9,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,2009,266,8,258,3.010


In [11]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 6.1: KIỂM TRA CÁC DÒNG THIẾU TOÀN BỘ THEO QUỐC GIA - CHỈ SỐ
# ─────────────────────────────────────────────────────────────

# Một dòng trong df_wide tương ứng với một cặp country_code - indicator_code.
# Nếu toàn bộ 23 năm đều thiếu thì dòng này không có giá trị phân tích trực tiếp.
rows_all_missing = df_wide.copy()
rows_all_missing["all_years_missing"] = rows_all_missing[YEAR_COLS_RAW].isna().all(axis=1)

all_missing_summary = (
    rows_all_missing[rows_all_missing["all_years_missing"]]
    [["Country Name", "Country Code", "Series Code", "Series Name"]]
    .rename(columns={
        "Country Name": "country_name",
        "Country Code": "country_code",
        "Series Code": "indicator_code",
        "Series Name": "indicator_name_en"
    })
    .sort_values(["indicator_code", "country_name"])
    .reset_index(drop=True)
)

print("--- Số dòng country-indicator thiếu toàn bộ các năm")
print(len(all_missing_summary))

print("\n--- Ví dụ các dòng thiếu toàn bộ")
display(all_missing_summary.head(30))

# Thống kê số indicator bị thiếu toàn bộ theo từng quốc gia
country_coverage_summary = (
    rows_all_missing
    .groupby(["Country Name", "Country Code"])
    .agg(
        total_indicators=("Series Code", "count"),
        indicators_all_years_missing=("all_years_missing", "sum")
    )
    .reset_index()
    .rename(columns={
        "Country Name": "country_name",
        "Country Code": "country_code"
    })
)

country_coverage_summary["available_indicator_ratio"] = (
    1 - country_coverage_summary["indicators_all_years_missing"] / country_coverage_summary["total_indicators"]
).round(3)

country_coverage_summary = country_coverage_summary.sort_values(
    ["indicators_all_years_missing", "country_name"],
    ascending=[False, True]
)

print("\n--- Quốc gia/vùng có nhiều chỉ số thiếu toàn bộ nhất")
display(country_coverage_summary.head(20))

--- Số dòng country-indicator thiếu toàn bộ các năm
273

--- Ví dụ các dòng thiếu toàn bộ


,country_name,country_code,indicator_code,indicator_name_en
0,Channel Islands,CHI,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...
1,Kosovo,XKX,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...
2,Monaco,MCO,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...
3,Not classified,INX,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...
4,San Marino,SMR,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...
5,St. Martin (French part),MAF,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...
6,Andorra,AND,EN.GHG.CO2.PC.CE.AR5,Carbon dioxide (CO2) emissions excluding LULUC...
7,Channel Islands,CHI,EN.GHG.CO2.PC.CE.AR5,Carbon dioxide (CO2) emissions excluding LULUC...
8,Curacao,CUW,EN.GHG.CO2.PC.CE.AR5,Carbon dioxide (CO2) emissions excluding LULUC...
9,Isle of Man,IMN,EN.GHG.CO2.PC.CE.AR5,Carbon dioxide (CO2) emissions excluding LULUC...



--- Quốc gia/vùng có nhiều chỉ số thiếu toàn bộ nhất


,country_name,country_code,total_indicators,indicators_all_years_missing,available_indicator_ratio
183,Not classified,INX,12,12,0.000
227,St. Martin (French part),MAF,12,10,0.167
44,Channel Islands,CHI,12,9,0.250
116,Isle of Man,IMN,12,8,0.333
5,American Samoa,ASM,12,7,0.417
79,Faroe Islands,FRO,12,7,0.417
84,French Polynesia,PYF,12,7,0.417
92,Greenland,GRL,12,7,0.417
94,Guam,GUM,12,7,0.417
127,Kosovo,XKX,12,7,0.417


## **7. TẠO BẢNG DỮ LIỆU SẠCH CHO TABLEAU**

### **7.1. Bảng long sạch**

Bảng `df_long_clean` là bảng chính khuyến nghị dùng trong Tableau. Bảng này:

* Ở dạng dài.
* Có một dòng cho mỗi tổ hợp quốc gia/vùng - chỉ số - năm.
* Chỉ giữ các dòng có giá trị quan sát thật.
* Có đầy đủ nhãn tiếng Việt, nhóm chỉ số, đơn vị đo và cờ đánh dấu nhóm tổng hợp.

Bảng này phù hợp cho hầu hết dashboard như:

* Biểu đồ đường theo thời gian.
* Biểu đồ bản đồ.
* Biểu đồ cột so sánh quốc gia.
* Biểu đồ heatmap theo năm và chỉ số.
* Dashboard có filter theo quốc gia, năm, nhóm chỉ số, loại đơn vị quan sát.

In [12]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 7: TẠO BẢNG LONG SẠCH
# ─────────────────────────────────────────────────────────────

df_long_clean = df_long.dropna(subset=["value"]).copy()

# Sắp xếp lại cột cho dễ đọc và dễ connect Tableau
long_cols_order = [
    "country_name",
    "country_code",
    "country_long_name",
    "country_short_name",
    "country_type",
    "is_aggregate",
    "region",
    "income_group",
    "currency_unit",
    "indicator_group",
    "indicator_code",
    "indicator_name_en",
    "indicator_name_vi",
    "short_name",
    "unit_vi",
    "unit_of_measure",
    "topic",
    "source",
    "periodicity",
    "reference_period",
    "aggregation_method",
    "year",
    "year_date",
    "period_group",
    "value",
    "data_status"
]

long_cols_order = [col for col in long_cols_order if col in df_long_clean.columns]
df_long_clean = df_long_clean[long_cols_order].sort_values(
    ["country_name", "indicator_code", "year"]
).reset_index(drop=True)

print("--- Kích thước bảng long đầy đủ, gồm cả missing")
print(df_long.shape)

print("\n--- Kích thước bảng long sạch, đã bỏ missing")
print(df_long_clean.shape)

print("\n--- Tỉ lệ dữ liệu còn lại sau khi bỏ missing")
print(f"{len(df_long_clean) / len(df_long) * 100:.2f}%")

print("\n--- 10 dòng đầu của bảng long sạch")
display(df_long_clean.head(10))

print("\n--- Kiểm tra trùng khóa country_code + indicator_code + year")
long_duplicate_keys = df_long_clean.duplicated(
    subset=["country_code", "indicator_code", "year"]
).sum()

print("Số khóa bị trùng:", long_duplicate_keys)

--- Kích thước bảng long đầy đủ, gồm cả missing
(73416, 31)

--- Kích thước bảng long sạch, đã bỏ missing
(58512, 26)

--- Tỉ lệ dữ liệu còn lại sau khi bỏ missing
79.70%

--- 10 dòng đầu của bảng long sạch


,country_name,country_code,country_long_name,country_short_name,country_type,is_aggregate,region,income_group,currency_unit,indicator_group,indicator_code,indicator_name_en,indicator_name_vi,short_name,unit_vi,unit_of_measure,topic,source,periodicity,reference_period,aggregation_method,year,year_date,period_group,value,data_status
0,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,Môi trường,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Năng lượng tái tạo,%,% (share) of final energy consumption,Environment: Energy production & use,"IEA Energy Statistics Data Browser, Internatio...",Annual,1990-2022,Weighted average,2000,2000-01-01,2000–2004,45.000,Có dữ liệu
1,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,Môi trường,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Năng lượng tái tạo,%,% (share) of final energy consumption,Environment: Energy production & use,"IEA Energy Statistics Data Browser, Internatio...",Annual,1990-2022,Weighted average,2001,2001-01-01,2000–2004,45.600,Có dữ liệu
2,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,Môi trường,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Năng lượng tái tạo,%,% (share) of final energy consumption,Environment: Energy production & use,"IEA Energy Statistics Data Browser, Internatio...",Annual,1990-2022,Weighted average,2002,2002-01-01,2000–2004,37.800,Có dữ liệu
3,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,Môi trường,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Năng lượng tái tạo,%,% (share) of final energy consumption,Environment: Energy production & use,"IEA Energy Statistics Data Browser, Internatio...",Annual,1990-2022,Weighted average,2003,2003-01-01,2000–2004,36.700,Có dữ liệu
4,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,Môi trường,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Năng lượng tái tạo,%,% (share) of final energy consumption,Environment: Energy production & use,"IEA Energy Statistics Data Browser, Internatio...",Annual,1990-2022,Weighted average,2004,2004-01-01,2000–2004,44.200,Có dữ liệu
5,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,Môi trường,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Năng lượng tái tạo,%,% (share) of final energy consumption,Environment: Energy production & use,"IEA Energy Statistics Data Browser, Internatio...",Annual,1990-2022,Weighted average,2005,2005-01-01,2005–2009,33.900,Có dữ liệu
6,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,Môi trường,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Năng lượng tái tạo,%,% (share) of final energy consumption,Environment: Energy production & use,"IEA Energy Statistics Data Browser, Internatio...",Annual,1990-2022,Weighted average,2006,2006-01-01,2005–2009,31.900,Có dữ liệu
7,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,Môi trường,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...


--- Kiểm tra trùng khóa country_code + indicator_code + year
Số khóa bị trùng: 0


### **7.2. Bảng wide theo chỉ số**

Ngoài bảng long, nhóm tạo thêm bảng wide theo chỉ số, trong đó mỗi dòng tương ứng với một quốc gia/vùng trong một năm, còn mỗi chỉ số là một cột riêng.

Bảng này phù hợp cho các biểu đồ cần so sánh quan hệ giữa nhiều chỉ số, ví dụ:

* Scatter plot giữa GDP bình quân đầu người và tuổi thọ.
* Scatter plot giữa chi tiêu y tế và tử vong trẻ em.
* So sánh phát thải CO2 và tỷ trọng năng lượng tái tạo.
* Phân tích tương quan giữa giáo dục và kinh tế.

Lưu ý: bảng wide có thể có nhiều giá trị thiếu hơn ở cấp dòng vì không phải quốc gia nào cũng có đủ 12 chỉ số trong cùng một năm.

In [13]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 7.2: TẠO BẢNG WIDE THEO CHỈ SỐ
# ─────────────────────────────────────────────────────────────

# Lookup tên cột ngắn cho từng indicator
field_name_lookup = indicator_info_vi.set_index("indicator_code")["tableau_field_name"].to_dict()

df_wide_for_tableau = (
    df_long_clean
    .pivot_table(
        index=[
            "country_name",
            "country_code",
            "country_long_name",
            "country_short_name",
            "country_type",
            "is_aggregate",
            "region",
            "income_group",
            "currency_unit",
            "year",
            "year_date",
            "period_group"
        ],
        columns="indicator_code",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)

# Đổi mã WDI thành tên cột dễ dùng trong Tableau
df_wide_for_tableau = df_wide_for_tableau.rename(columns=field_name_lookup)

# Xóa tên trục columns do pivot_table tạo ra
df_wide_for_tableau.columns.name = None

# Sắp xếp cột
base_cols = [
    "country_name",
    "country_code",
    "country_long_name",
    "country_short_name",
    "country_type",
    "is_aggregate",
    "region",
    "income_group",
    "currency_unit",
    "year",
    "year_date",
    "period_group"
]

indicator_wide_cols = [
    info["tableau_field_name"]
    for info in INDICATOR_INFO_VI.values()
    if info["tableau_field_name"] in df_wide_for_tableau.columns
]

df_wide_for_tableau = df_wide_for_tableau[base_cols + indicator_wide_cols]

print("--- Kích thước bảng wide cho Tableau")
print(df_wide_for_tableau.shape)

print("\n--- 10 dòng đầu")
display(df_wide_for_tableau.head(10))

print("\n--- Tỉ lệ missing của các cột chỉ số trong bảng wide")
wide_missing = (
    df_wide_for_tableau[indicator_wide_cols]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .reset_index()
)

wide_missing.columns = ["indicator_field", "missing_pct"]
display(wide_missing)

--- Kích thước bảng wide cho Tableau
(4945, 24)

--- 10 dòng đầu


,country_name,country_code,country_long_name,country_short_name,country_type,is_aggregate,region,income_group,currency_unit,year,year_date,period_group,gdp_per_capita,gdp_per_capita_growth_pct,poverty_headcount_pct,life_expectancy_years,health_expenditure_pc_usd,under5_mortality_per_1000,co2_pc_tco2e,renewable_energy_pct,school_primary_gross_pct,school_secondary_gross_pct,school_tertiary_gross_pct,education_gpi
0,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,2000,2000-01-01,2000–2004,308.318,NaN,NaN,55.005,NaN,131.700,0.050,45.000,21.458,NaN,NaN,NaN
1,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,2001,2001-01-01,2000–2004,277.118,-10.119,NaN,55.511,NaN,127.500,0.046,45.600,22.140,14.040,NaN,0.000
2,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,2002,2002-01-01,2000–2004,338.140,22.020,NaN,56.225,16.707,123.100,0.044,37.800,73.401,NaN,NaN,NaN
3,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,2003,2003-01-01,2000–2004,346.072,2.346,NaN,57.171,17.746,118.500,0.045,36.700,95.336,13.960,1.374,0.544
4,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,2004,2004-01-01,2000–2004,338.637,-2.148,NaN,57.810,21.423,113.900,0.038,44.200,105.475,19.214,1.391,0.406
5,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,2005,2005-01-01,2005–2009,363.640,7.383,NaN,58.247,25.114,109.300,0.052,33.900,99.504,20.274,NaN,0.548
6,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,2006,2006-01-01,2005–2009,367.758,1.132,NaN,58.553,28.941,104.800,0.056,31.900,102.285,29.524,NaN,0.573
7,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,2007,2007-01-01,2005–2009,410.758,11.692,33.700,58.956,32.709,100.400,0.080,28.800,99.323,28.815,NaN,0.567
8,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,2008,2008-01-01,2005–2009,417.647,1.677,NaN,59.708,39.886,96.100,0.147,21.200,104.445,38.931,NaN,0.586
9,Afghanistan,AFG,Islamic State of Afghanistan,Afghanistan,Quốc gia/vùng lãnh thổ,False,Middle East & North Africa,Low income,Afghan afghani,2009,2009-01-01,2005–2009,488.831,17.044,NaN,60.248,43.134,91.900,0.224,16.500,99.705,44.300,3.938,0.610



--- Tỉ lệ missing của các cột chỉ số trong bảng wide


,indicator_field,missing_pct
0,poverty_headcount_pct,79.470
1,education_gpi,42.570
2,school_tertiary_gross_pct,40.530
3,school_secondary_gross_pct,33.290
4,school_primary_gross_pct,23.460
5,health_expenditure_pc_usd,12.680
6,under5_mortality_per_1000,9.770
7,co2_pc_tco2e,6.510
8,renewable_energy_pct,5.800
9,gdp_per_capita,4.730


## **8. KIỂM TRA GIÁ TRỊ BẤT THƯỜNG**

### **8.1. Kiểm tra min-max theo từng chỉ số**

Với dữ liệu WDI, nhiều giá trị cực trị không nhất thiết là lỗi. Ví dụ:

* GDP bình quân đầu người rất cao ở một số nền kinh tế nhỏ hoặc nhóm thu nhập cao.
* Phát thải CO2 bình quân đầu người có thể rất cao ở các quốc gia khai thác dầu khí.
* Tỷ lệ nhập học gross có thể vượt 100% vì chỉ số này tính tổng số học sinh nhập học bất kể tuổi.
* Tăng trưởng GDP có thể âm mạnh trong khủng hoảng hoặc dương rất cao khi nền kinh tế phục hồi từ nền thấp.

Do đó, nhóm không loại ngoại lai bằng IQR một cách máy móc. Thay vào đó, nhóm thống kê min-max để phục vụ kiểm tra và diễn giải trong báo cáo.

In [14]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 8: THỐNG KÊ MIN-MAX VÀ KIỂM TRA GIÁ TRỊ BẤT THƯỜNG
# ─────────────────────────────────────────────────────────────

indicator_value_summary = (
    df_long_clean
    .groupby([
        "indicator_code",
        "indicator_name_vi",
        "unit_vi"
    ])
    .agg(
        count=("value", "count"),
        min_value=("value", "min"),
        q1=("value", lambda s: s.quantile(0.25)),
        median=("value", "median"),
        mean=("value", "mean"),
        q3=("value", lambda s: s.quantile(0.75)),
        max_value=("value", "max")
    )
    .reset_index()
)

numeric_cols = ["min_value", "q1", "median", "mean", "q3", "max_value"]
indicator_value_summary[numeric_cols] = indicator_value_summary[numeric_cols].round(3)

print("--- Thống kê giá trị theo từng chỉ số")
display(indicator_value_summary)

# Lấy top/bottom 5 giá trị theo từng chỉ số để kiểm tra nhanh
extreme_records = []

for code in df_long_clean["indicator_code"].unique():
    temp = df_long_clean[df_long_clean["indicator_code"] == code].copy()
    
    bottom = temp.nsmallest(5, "value")
    bottom["extreme_type"] = "Nhỏ nhất"
    
    top = temp.nlargest(5, "value")
    top["extreme_type"] = "Lớn nhất"
    
    extreme_records.append(pd.concat([bottom, top], ignore_index=True))

df_extreme_values = pd.concat(extreme_records, ignore_index=True)

df_extreme_values = df_extreme_values[
    [
        "indicator_code",
        "indicator_name_vi",
        "extreme_type",
        "country_name",
        "country_code",
        "country_type",
        "year",
        "value",
        "unit_vi"
    ]
].sort_values(["indicator_code", "extreme_type", "value"])

print("\n--- Top/bottom giá trị để kiểm tra bất thường")
display(df_extreme_values.head(60))

--- Thống kê giá trị theo từng chỉ số


,indicator_code,indicator_name_vi,unit_vi,count,min_value,q1,median,mean,q3,max_value
0,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,%,5710,0.000,6.500,20.200,30.411,50.600,98.300
1,EN.GHG.CO2.PC.CE.AR5,Phát thải CO2 bình quân đầu người (tCO2e/người),tCO2e/người,5773,0.000,0.658,2.461,4.734,6.188,202.865
2,NY.GDP.PCAP.KD,GDP bình quân đầu người (USD cố định 2015),USD cố định 2015,5861,233.032,"1,747.718","5,272.718","14,288.470","17,980.730","214,359.606"
3,NY.GDP.PCAP.KD.ZG,Tăng trưởng GDP bình quân đầu người (%),%,5868,-55.294,0.170,2.233,2.085,4.316,91.781
4,SE.ENR.PRSC.FM.ZS,Chỉ số cân bằng giới trong nhập học tiểu học v...,Tỷ lệ nữ/nam,3837,0.000,0.960,0.995,0.972,1.013,1.176
5,SE.PRM.ENRR,"Tỷ lệ nhập học tiểu học, gross (%)",%,4881,21.458,97.088,101.444,101.398,106.856,183.989
6,SE.SEC.ENRR,"Tỷ lệ nhập học trung học, gross (%)",%,4390,5.941,60.019,86.050,79.300,99.463,164.080
7,SE.TER.ENRR,"Tỷ lệ nhập học bậc đại học/cao đẳng, gross (%)",%,4000,0.116,12.869,31.406,37.399,58.927,166.390
8,SH.DYN.MORT,Tử vong trẻ em dưới 5 tuổi (trên 1.000 ca sinh...,Trên 1.000 ca sinh sống,5612,1.400,9.800,23.300,39.708,58.712,489.300
9,SH.XPD.CHEX.PC.CD,Chi tiêu y tế bình quân đầu người (USD hiện hành),USD hiện hành,5450,4.176,63.583,247.237,937.595,803.824,"12,586.196"



--- Top/bottom giá trị để kiểm tra bất thường


,indicator_code,indicator_name_vi,extreme_type,country_name,country_code,country_type,year,value,unit_vi
8,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Lớn nhất,"Congo, Dem. Rep.",COD,Quốc gia/vùng lãnh thổ,2000,97.900,%
9,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Lớn nhất,"Congo, Dem. Rep.",COD,Quốc gia/vùng lãnh thổ,2004,97.900,%
7,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Lớn nhất,"Congo, Dem. Rep.",COD,Quốc gia/vùng lãnh thổ,2003,98.000,%
5,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Lớn nhất,"Congo, Dem. Rep.",COD,Quốc gia/vùng lãnh thổ,2001,98.300,%
6,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Lớn nhất,"Congo, Dem. Rep.",COD,Quốc gia/vùng lãnh thổ,2002,98.300,%
0,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Nhỏ nhất,American Samoa,ASM,Quốc gia/vùng lãnh thổ,2000,0.000,%
1,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Nhỏ nhất,American Samoa,ASM,Quốc gia/vùng lãnh thổ,2001,0.000,%
2,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Nhỏ nhất,American Samoa,ASM,Quốc gia/vùng lãnh thổ,2002,0.000,%
3,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Nhỏ nhất,American Samoa,ASM,Quốc gia/vùng lãnh thổ,2003,0.000,%
4,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Nhỏ nhất,American Samoa,ASM,Quốc gia/vùng lãnh thổ,2004,0.000,%


## **9. TẠO CÁC BẢNG PHỤ TRỢ CHO DASHBOARD**

### **9.1. Bảng danh mục quốc gia và bảng danh mục chỉ số**

Để thiết kế dashboard trong Tableau gọn hơn, nhóm tạo thêm các bảng phụ trợ:

* `country_dimension`: danh sách quốc gia/vùng/tổng hợp.
* `indicator_dimension`: danh sách chỉ số, tên tiếng Việt, nhóm chỉ số và đơn vị đo.
* `data_quality_indicator`: tóm tắt chất lượng dữ liệu theo chỉ số.
* `data_quality_year`: tóm tắt chất lượng dữ liệu theo năm.

Các bảng này có thể dùng để viết báo cáo hoặc connect thêm vào Tableau nếu cần.

In [15]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 9: TẠO DIMENSION TABLES VÀ BẢNG CHẤT LƯỢNG DỮ LIỆU
# ─────────────────────────────────────────────────────────────

country_dimension = (
    df_long[[
        "country_name",
        "country_code",
        "country_long_name",
        "country_short_name",
        "country_type",
        "is_aggregate",
        "region",
        "income_group",
        "currency_unit"
    ]]
    .drop_duplicates()
    .sort_values(["country_type", "region", "income_group", "country_name"])
    .reset_index(drop=True)
)

indicator_dimension = (
    indicator_metadata[[
        "indicator_code",
        "indicator_name_en",
        "indicator_name_vi",
        "indicator_group",
        "unit_vi",
        "short_name",
        "tableau_field_name"
    ]]
    .drop_duplicates()
    .sort_values(["indicator_group", "indicator_code"])
    .reset_index(drop=True)
)

data_quality_indicator = missing_by_indicator.copy()
data_quality_year = missing_by_year.copy()
data_quality_indicator_year = missing_by_indicator_year.copy()

print("--- Country dimension")
display(country_dimension.head(20))

print("\n--- Indicator dimension")
display(indicator_dimension)

print("\n--- Data quality by indicator")
display(data_quality_indicator)

--- Country dimension


,country_name,country_code,country_long_name,country_short_name,country_type,is_aggregate,region,income_group,currency_unit
0,Africa Eastern and Southern,AFE,Africa Eastern and Southern,Africa Eastern and Southern,Nhóm tổng hợp/khu vực,True,NaN,NaN,NaN
1,Africa Western and Central,AFW,Africa Western and Central,Africa Western and Central,Nhóm tổng hợp/khu vực,True,NaN,NaN,NaN
2,Arab World,ARB,Arab World,Arab World,Nhóm tổng hợp/khu vực,True,NaN,NaN,NaN
3,Caribbean small states,CSS,Caribbean small states,Caribbean small states,Nhóm tổng hợp/khu vực,True,NaN,NaN,NaN
4,Central Europe and the Baltics,CEB,Central Europe and the Baltics,Central Europe and the Baltics,Nhóm tổng hợp/khu vực,True,NaN,NaN,NaN
5,Early-demographic dividend,EAR,Early-demographic dividend,Early-demographic dividend,Nhóm tổng hợp/khu vực,True,NaN,NaN,NaN
6,East Asia & Pacific,EAS,East Asia & Pacific,East Asia & Pacific,Nhóm tổng hợp/khu vực,True,NaN,NaN,NaN
7,East Asia & Pacific (IDA & IBRD countries),TEA,East Asia & Pacific (IDA & IBRD),East Asia & Pacific (IDA & IBRD),Nhóm tổng hợp/khu vực,True,NaN,NaN,NaN
8,East Asia & Pacific (excluding high income),EAP,East Asia & Pacific (excluding high income),East Asia & Pacific (excluding high income),Nhóm tổng hợp/khu vực,True,NaN,NaN,NaN
9,Euro area,EMU,Euro area,Euro area,Nhóm tổng hợp/khu vực,True,NaN,NaN,NaN



--- Indicator dimension


,indicator_code,indicator_name_en,indicator_name_vi,indicator_group,unit_vi,short_name,tableau_field_name
0,SE.ENR.PRSC.FM.ZS,"School enrollment, primary and secondary (gros...",Chỉ số cân bằng giới trong nhập học tiểu học v...,Giáo dục,Tỷ lệ nữ/nam,GPI giáo dục,education_gpi
1,SE.PRM.ENRR,"School enrollment, primary (% gross)","Tỷ lệ nhập học tiểu học, gross (%)",Giáo dục,%,Nhập học tiểu học,school_primary_gross_pct
2,SE.SEC.ENRR,"School enrollment, secondary (% gross)","Tỷ lệ nhập học trung học, gross (%)",Giáo dục,%,Nhập học trung học,school_secondary_gross_pct
3,SE.TER.ENRR,"School enrollment, tertiary (% gross)","Tỷ lệ nhập học bậc đại học/cao đẳng, gross (%)",Giáo dục,%,Nhập học bậc cao,school_tertiary_gross_pct
4,NY.GDP.PCAP.KD,GDP per capita (constant 2015 US$),GDP bình quân đầu người (USD cố định 2015),Kinh tế,USD cố định 2015,GDP/người,gdp_per_capita
5,NY.GDP.PCAP.KD.ZG,GDP per capita (annual % growth),Tăng trưởng GDP bình quân đầu người (%),Kinh tế,%,Tăng trưởng GDP/người,gdp_per_capita_growth_pct
6,SI.POV.NAHC,Poverty headcount ratio at national poverty li...,Tỷ lệ nghèo theo chuẩn quốc gia (%),Kinh tế - Xã hội,%,Tỷ lệ nghèo,poverty_headcount_pct
7,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Môi trường,%,Năng lượng tái tạo,renewable_energy_pct
8,EN.GHG.CO2.PC.CE.AR5,Carbon dioxide (CO2) emissions excluding LULUC...,Phát thải CO2 bình quân đầu người (tCO2e/người),Môi trường,tCO2e/người,CO2/người,co2_pc_tco2e
9,SH.DYN.MORT,"Mortality rate, under-5 (per 1,000 live births)",Tử vong trẻ em dưới 5 tuổi (trên 1.000 ca sinh...,Sức khỏe,Trên 1.000 ca sinh sống,Tử vong dưới 5 tuổi,under5_mortality_per_1000



--- Data quality by indicator


,indicator_code,indicator_name_vi,indicator_group,total_observations,missing_observations,available_observations,missing_pct
10,SI.POV.NAHC,Tỷ lệ nghèo theo chuẩn quốc gia (%),Kinh tế - Xã hội,6118,5083,1035,83.080
4,SE.ENR.PRSC.FM.ZS,Chỉ số cân bằng giới trong nhập học tiểu học v...,Giáo dục,6118,2281,3837,37.280
7,SE.TER.ENRR,"Tỷ lệ nhập học bậc đại học/cao đẳng, gross (%)",Giáo dục,6118,2118,4000,34.620
6,SE.SEC.ENRR,"Tỷ lệ nhập học trung học, gross (%)",Giáo dục,6118,1728,4390,28.240
5,SE.PRM.ENRR,"Tỷ lệ nhập học tiểu học, gross (%)",Giáo dục,6118,1237,4881,20.220
9,SH.XPD.CHEX.PC.CD,Chi tiêu y tế bình quân đầu người (USD hiện hành),Sức khỏe,6118,668,5450,10.920
8,SH.DYN.MORT,Tử vong trẻ em dưới 5 tuổi (trên 1.000 ca sinh...,Sức khỏe,6118,506,5612,8.270
0,EG.FEC.RNEW.ZS,Tỷ trọng năng lượng tái tạo trong tiêu thụ năn...,Môi trường,6118,408,5710,6.670
1,EN.GHG.CO2.PC.CE.AR5,Phát thải CO2 bình quân đầu người (tCO2e/người),Môi trường,6118,345,5773,5.640
2,NY.GDP.PCAP.KD,GDP bình quân đầu người (USD cố định 2015),Kinh tế,6118,257,5861,4.200


## **10. XUẤT DỮ LIỆU SAU TIỀN XỬ LÝ**

### **10.1. Các file đầu ra dùng cho Tableau**

Sau khi tiền xử lý, nhóm chỉ xuất các file dữ liệu cần thiết để kết nối vào Tableau và xây dựng dashboard.

| File | Mục đích |
| :--- | :--- |
| `wdi_long_clean.csv` | File chính dùng để vẽ biểu đồ trong Tableau, dạng long, đã loại bỏ giá trị thiếu |
| `wdi_wide_for_tableau.csv` | File phụ dùng cho các biểu đồ so sánh/tương quan giữa nhiều chỉ số |

File khuyến nghị dùng chính trong Tableau là `wdi_long_clean.csv`.

File `wdi_wide_for_tableau.csv` chỉ dùng khi cần vẽ các biểu đồ như scatter plot giữa GDP bình quân đầu người và tuổi thọ, chi tiêu y tế và tử vong trẻ em, hoặc CO2 và năng lượng tái tạo.

In [16]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 10: XUẤT FILE CSV DÙNG CHO TABLEAU
# ─────────────────────────────────────────────────────────────

output_files = {
    "wdi_long_clean.csv": df_long_clean,
    "wdi_wide_for_tableau.csv": df_wide_for_tableau
}

for filename, df_out in output_files.items():
    output_path = OUTPUT_DIR / filename
    df_out.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Đã xuất: {output_path} | shape = {df_out.shape}")

print("\nHoàn tất xuất dữ liệu dùng cho Tableau.")

Đã xuất: d:\BaiTapKi2Nam3\TQHDL\Lab\Lab02\data\processed\wdi_long_clean.csv | shape = (58512, 26)
Đã xuất: d:\BaiTapKi2Nam3\TQHDL\Lab\Lab02\data\processed\wdi_wide_for_tableau.csv | shape = (4945, 24)

Hoàn tất xuất dữ liệu dùng cho Tableau.


## **11. TỔNG KẾT QUY TRÌNH TIỀN XỬ LÝ**

Quy trình tiền xử lý đã hoàn thành các bước chính sau:

1. Đọc dữ liệu chính từ `Data.csv`.
2. Đọc metadata sạch từ file `Metadata.xlsx`, gồm metadata chỉ số và metadata quốc gia/vùng lãnh thổ.
3. Loại bỏ các dòng footer và dòng trống không phải quan sát dữ liệu.
4. Chuẩn hóa ký hiệu missing `..` thành `NaN`.
5. Ép kiểu dữ liệu các cột năm sang dạng số.
6. Chuẩn hóa metadata chỉ số từ sheet `Series - Metadata`.
7. Chuẩn hóa metadata quốc gia/vùng lãnh thổ từ sheet `Country - Metadata`.
8. Việt hóa tên chỉ số và phân nhóm chỉ số theo các chủ đề: kinh tế, sức khỏe, môi trường, giáo dục.
9. Chuyển dữ liệu từ dạng rộng sang dạng dài để phù hợp với Tableau.
10. Bổ sung thông tin khu vực, nhóm thu nhập và đơn vị tiền tệ cho từng quốc gia/vùng lãnh thổ.
11. Đánh dấu các nhóm tổng hợp của World Bank bằng cột `is_aggregate`.
12. Thống kê chất lượng dữ liệu theo chỉ số, theo năm và theo quốc gia.
13. Tạo bảng long phục vụ dashboard, trực quan hóa.

### **Lưu ý khi phân tích**

* Không nên tự ý điền giá trị thiếu cho dữ liệu WDI vì có thể làm sai lệch xu hướng phát triển.
* Khi phân tích tỷ lệ nghèo, cần chú ý vì chỉ số này thường thiếu nhiều dữ liệu hơn các chỉ số còn lại.
* Khi xếp hạng quốc gia, nên lọc `country_type = "Quốc gia/vùng lãnh thổ"` để loại các nhóm tổng hợp như `World`, `High income`, `OECD members`.
* Khi cần so sánh với trung bình khu vực hoặc thế giới, có thể giữ lại `country_type = "Nhóm tổng hợp/khu vực"`.
* Có thể dùng thêm `region` và `income_group` làm filter trong Tableau để dashboard chuyên nghiệp hơn.
* File chính nên dùng trong Tableau là `wdi_long_clean.csv`.